<a href="https://colab.research.google.com/github/KellySouza3110/Projeto-Financeiro/blob/main/Sistema_Cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.core.interactiveshell import Instance
!pip install flask requests
from flask import Flask, request, jsonify
from threading import Thread
import sqlite3
import requests
import time

# 1. CONFIGURACAO DO SERVIDOR
app = Flask(__name__)
BANCO = "escola_cloud.db"
URL = "http://127.0.0.1:5000"

# 2. CRIACAO DO BANCO DE DADOS
def conectar():
    return sqlite3.connect(BANCO, timeout=10)
def criar_banco():
    with conectar() as banco:
        banco.execute("""
            CREATE TABLE IF NOT EXISTS alunos(
                codigo INTEGER PRIMARY KEY AUTOINCREMENT,
                nome TEXT NOT NULL,
                nota1 REAL NOT NULL,
                nota2 REAL NOT NULL,
                media REAL NOT NULL,
                situacao TEXT NOT NULL
              )
        """)
criar_banco()

# 3.VALIDACAO DOS DADOS NO SERVIDOR
def validar_aluno(dados):
    if not isinstance(dados, dict):
        return None, "JSON invalido"
    nome = dados.get("nome")
    nota1 = dados.get("nota1")
    nota2 = dados.get("nota2")
    if not isinstance(nome, str) or not nome.strip():
        return None, "Nome invalido"
    for nota in (nota1, nota2):
        if isinstance(nota, bool) or not isinstance(nota, (int, float)) or not (0 <= nota <= 10):
            return None, "Nota invalida. Use valores de 0 a 10"
    media = round((nota1 + nota2) / 2, 2)
    situacao = "Aprovado" if media >= 6 else "Reprovado"
    return {
        "nome": nome.strip(),
        "nota1": nota1,
        "nota2": nota2,
        "media": media,
        "situacao": situacao
    }, None

# 4. ROTA INICIAL
@app.route("/", methods={"GET"})
def inicio():
    return jsonify({"mensagem": "Servidor Cloud funcionando!", "banco": BANCO}), 200

# 5. CADASTRAR ALUNO
@app.route("/cadastrar", methods={"POST"})
def cadastrar():
    dados, erro = validar_aluno(request.get_json(silent=True))
    if erro:
        return jsonify({"erro": erro}), 400
    try:
        with conectar() as banco:
            cursor = banco.execute("""
                INSERT INTO alunos
                (nome, nota1, nota2, media, situacao)
                VALUES (?, ?, ?, ?, ?)
            """, (
                dados["nome"], dados["nota1"],
                dados["nota2"], dados["media"],
                dados["situacao"]
            ))
            codigo = cursor.lastrowid
            return jsonify({"mensagem": "Aluno Cadastrado!", "codigo": codigo, "dados": dados}), 201
    except sqlite3.Error:
        return jsonify({"erro": "Erro no banco de dados"}), 500

# 6. LISTAR ALUNOS
@app.route("/alunos", methods={"GET"})
def listar():
    try:
        with conectar() as banco:
            banco.row_factory = sqlite3.Row
            registros = banco.execute("SELECT * FROM alunos ORDER BY codigo").fetchall()
        return jsonify([dict(a) for a in registros]), 200
    except sqlite3.Error:
        return jsonify({"erro": "Erro ao listar alunos"}), 500

# 7. CONSULTAR ALUNO
@app.route("/alunos/<int:codigo>", methods={"GET"})
def consultar(codigo):
    try:
        with conectar() as banco:
            banco.row_factory = sqlite3.Row
            aluno = banco.execute("SELECT * FROM alunos WHERE codigo = ?", (codigo,)).fetchone()
        if aluno is None:
            return jsonify({"erro": "Aluno nao encontrado"}), 404
        return jsonify(dict(aluno)), 200
    except sqlite3.Error:
        return jsonify({"erro": "Erro na consulta"}), 500

# 8. ALTERAR ALUNO
@app.route("/alunos/<int:codigo>", methods={"PUT"})
def alterar(codigo):
    dados, erro = validar_aluno(request.get_json(silent=True))
    if erro:
        return jsonify({"erro": erro}), 400
    try:
        with conectar() as banco:
            cursor = banco.execute("""
                UPDATE alunos
                SET nome = ?, nota1 = ?, nota2 = ?, media = ?, situacao = ?
                WHERE codigo = ?
            """, (
                dados["nome"], dados["nota1"],
                dados["nota2"], dados["media"],
                dados["situacao"], codigo
            ))
            if cursor.rowcount == 0:
                return jsonify({"erro": "Aluno não encontrado"}), 404
            return jsonify({"mensagem": "Aluno atualizado!", "codigo": codigo, "dados": dados}), 200
    except sqlite3.Error:
        return jsonify({"erro": "Erro ao alterar aluno"}), 500

# 9. EXCLUIR ALUNO
@app.route("/alunos/<int:codigo>", methods=["DELETE"])
def excluir(codigo):
    try:
        with conectar() as banco:
            cursor = banco.execute("DELETE FROM alunos WHERE codigo = ?", (codigo,))
            if cursor.rowcount == 0:
                return jsonify({"erro": "Aluno nao encontrado"}), 404
        return jsonify({"mensagem": "Aluno excluido"}), 200
    except sqlite3.Error:
        return jsonify({"erro": "Erro ao excluir aluno"}), 500
# 10. INICIAR SERVIDOR EM SEGUNDO PLANO
def iniciar_servidor():
    try:
        app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)
    except Exception:
        pass
servidor = Thread(target=iniciar_servidor, daemon=True)
servidor.start()
time.sleep(2)
# 11. COMUNICACAO CLIENTE SERVIDOR
def enviar(metodo, rota, dados=None):
    try:
        resposta = requests.request(method=metodo, url=URL + rota, json=dados, timeout=5)
        print("\nSTATUS HTTP:", resposta.status_code)
        try:
            resultado = resposta.json()
        except ValueError:
            print("Erro: resposta JSON invalida.")
            return None
        if resposta.status_code >= 400:
            print("ERRO:", resultado)
        return resultado
    except requests.exceptions.timeout:
        print("Erro: servidor demorou para responder.")
    except requests.exceptions.ConnectionError:
        print("Erro: falha na conexao com o servidor")
    except requests.exceptions.RequestException as erro:
        print("Erro de comunicacao:", erro)
    return None
# 12. VALIDACAO DA DIGITACAO
def ler_nota(mensagem):
    while True:
        try:
            nota = float(input(mensagem).replace(",", "."))
            if 0 <= nota <= 10:
                return nota
            print("Erro: nota deve estar entre 0 e 10.")
        except ValueError:
            print("Erro: digite um numero valido.")

def ler_codigo():
    while True:
        try:
            codigo = int(input("Código do aluno: ").strip())
            return codigo
        except ValueError:
            print("Erro: digite um código numérico válido.")

def ler_aluno():
    while True:
        nome = input("Nome do aluno: ").strip()
        if nome:
            break
        print("Erro: nome nao pode ficar vazio.")
    nota1 = ler_nota("Primeira nota: ")
    nota2 = ler_nota("Segunda nota: ")
    return {
        "nome": nome,
        "nota1": nota1,
        "nota2": nota2
    }
# 13. EXIBIR ALUNOS
def mostrar_alunos():
    resultado = enviar("GET", "/alunos")
    if not isinstance(resultado, list):
        return
    if not resultado:
        print("Nenhum aluno cadastrado.")
        return
    print("\n===== ALUNOS CADASTRADOS =====

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
INFO:werkzeug:127.0.0.1 - - [09/Nov/2026 02:16:12] "GET / HTTP/1.1" 200 -



SERVIDOR INICIADO!

STATUS HTTP: 200

======== SISTEMA CLOUD ========
1 - Cadastrar aluno
2 - Listar aluno
3 - Consultar aluno
4 - Alterar aluno
5 - Excluir aluno
6 - Testar conexao
0 - Sair
